In [0]:
# Databricks notebook source
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

STORAGE = "stecomlakedev"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"
GOLD = f"abfss://gold@{STORAGE}.dfs.core.windows.net"
DQ = f"{GOLD}/_dq_results"

RUN_TS = datetime.now(timezone.utc)

DQ_SCHEMA = StructType([
    StructField("run_ts", TimestampType()),
    StructField("check", StringType()),
    StructField("failed_rows", LongType()),
    StructField("status", StringType()),
])

def check(name, condition, df):
    failed = df.filter(~condition).count()
    status = "PASS" if failed == 0 else "FAIL"
    print(f"[{status}] {name} — failing rows: {failed}")
    return (RUN_TS, name, failed, status)

def result(name, failed):
    status = "PASS" if failed == 0 else "FAIL"
    print(f"[{status}] {name} — failing rows: {failed}")
    return (RUN_TS, name, failed, status)

# COMMAND ----------
fact = spark.read.format("delta").load(f"{GOLD}/fact_orders")
dimc = spark.read.format("delta").load(f"{GOLD}/dim_customer")
dimp = spark.read.format("delta").load(f"{GOLD}/dim_product")
silver_events = spark.read.format("delta").load(f"{SILVER}/clickstream_events")

results = []

# COMMAND ----------
results += [
    check("fact_orders.price >= 0", F.col("price") >= 0, fact),
    check("fact_orders.order_id not null", F.col("order_id").isNotNull(), fact),
    check("fact_orders.order_date not null", F.col("order_date").isNotNull(), fact),
    check("dim_customer one current row per id",
          F.col("cnt") == 1,
          dimc.filter("is_current").groupBy("customer_id").agg(F.count("*").alias("cnt"))),
    check("dim_product unique product_id",
          F.col("cnt") == 1,
          dimp.groupBy("product_id").agg(F.count("*").alias("cnt"))),
    check("clickstream user_id not null", F.col("user_id").isNotNull(), silver_events),
]

# COMMAND ----------
results.append(result(
    "fact_orders -> dim_customer RI",
    fact.join(dimc.select("customer_sk").distinct(), "customer_sk", "left_anti").count()))

results.append(result(
    "fact_orders -> dim_product RI",
    fact.join(dimp.select("product_id").distinct(), "product_id", "left_anti").count()))

results.append(result(
    "fact_orders unknown customer rate < 0.1%",
    0 if fact.filter("customer_sk = -1").count() / fact.count() < 0.001 else 1))

# COMMAND ----------
dq = spark.createDataFrame(results, DQ_SCHEMA)
display(dq)

dq.write.format("delta").mode("append").option("mergeSchema", "true").save(DQ)

failures = [r for r in results if r[3] == "FAIL"]
if failures:
    raise Exception(f"Data quality checks FAILED: {[r[1] for r in failures]}")
print("all checks passed")

# COMMAND ----------
display(spark.read.format("delta").load(DQ).orderBy(F.col("run_ts").desc()))